# Neural Machine Translation

A neural machine translation model based on the sequence-to-sequence (seq2seq) models proposed by Sutskever et al., 2014 and Cho et al., 2014. The seq2seq model is widely used in Machine Translation systems such as Google’s neural machine translation system (GNMT) (Wu et al., 2016).

For training and evaluating our mode, we will use the English-Vietnamese parallel corpus of TED talks provided by the IWSLT Evaluation Campaign. We will translate from Vietnamese into English.

The parallel corpus:
1. **data.30.vi** - a file where each line contains a Vietnamese sentence to be translated (i.e. the source sentences)
2. **data.30.en** - a file where each line contains an English sentence corresponding to the Vietnamese sentence in the same line position. (i.e. the target sentences)


In [1]:
!wget 'https://github.com/juntaoy/ECS7001_LAB_DATASETS/raw/refs/heads/main/NMT_data.zip'
!unzip NMT_data.zip -x __MACOSX/*

In [2]:
!pip install sacrebleu

  Using cached sacrebleu-2.5.1-py3-none-any.whl.metadata (51 kB)
  Using cached portalocker-3.1.1-py3-none-any.whl.metadata (8.6 kB)
  Using cached regex-2024.11.6-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (40 kB)
  Using cached tabulate-0.9.0-py3-none-any.whl.metadata (34 kB)
  Using cached lxml-5.3.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (3.7 kB)
Using cached sacrebleu-2.5.1-py3-none-any.whl (104 kB)
Using cached tabulate-0.9.0-py3-none-any.whl (35 kB)
Using cached lxml-5.3.1-cp311-cp311-manylinux_2_28_x86_64.whl (5.0 MB)
Using cached portalocker-3.1.1-py3-none-any.whl (19 kB)
Using cached regex-2024.11.6-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (792 kB)

[notice] A new release of pip is available: 24.1.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import collections
import numpy as np
import time
from sacrebleu import corpus_bleu

SOURCE_PATH = 'data.30.vi'
TARGET_PATH = 'data.30.en'

## The `LanguageDict` class stores the language resources

In [4]:
class LanguageDict():
    def __init__(self, sents):
        word_counter = collections.Counter(tok.lower() for sent in sents for tok in sent)
        
        self.vocab = []
        self.vocab.append('<pad>') #zero paddings
        self.vocab.append('<unk>')
        # add only words that appear at least 10 times in the corpus
        self.vocab.extend([t for t,c in word_counter.items() if c > 10])
    
        self.word2ids = {w:id for id, w in enumerate(self.vocab)}
        self.UNK = self.word2ids['<unk>']
        self.PAD = self.word2ids['<pad>']

In [5]:
sentences = [["hello", "world", "hello", "hello"], ["hello", "hello", "this", "is", "a", "test"], ["hello", "this", "is", "hello", "hello", "hello", "hello", "hello"]]
lang_dict = LanguageDict(sentences)
# print(lang_dict.word_counter)
print(lang_dict.vocab)
print(lang_dict.word2ids)

['<pad>', '<unk>', 'hello']
{'<pad>': 0, '<unk>': 1, 'hello': 2}


In [6]:
def pad_sequences(seq_list, max_len=None, pad_value=0):
    """
    A simple PyTorch-like pad_sequences function.
    seq_list: List of lists of token IDs.
    max_len : If None, will use the length of the longest sequence.
    pad_value: ID to use for padding.
    Returns a 2D NumPy array with shape [batch_size, max_length].
    """
    if max_len is None:
        max_len = max(len(seq) for seq in seq_list)
    padded = []
    for seq in seq_list:
        seq = seq[:max_len]
        padded.append(seq + [pad_value]*(max_len - len(seq)))
    return np.array(padded)

In [7]:
sentences = [["hello", "world", "hello", "hello"], ["hello", "hello", "this", "is", "a", "test"], ["hello", "this", "is", "hello", "hello", "hello", "hello", "hello"]]
pad_seq = pad_sequences(sentences)
print(pad_seq)

[['hello' 'world' 'hello' 'hello' '0' '0' '0' '0']
 ['hello' 'hello' 'this' 'is' 'a' 'test' '0' '0']
 ['hello' 'this' 'is' 'hello' 'hello' 'hello' 'hello' 'hello']]


The method reads the given file and loads the first max_num_examples sentences and split them into train/dev/test dataset:

In [8]:
def load_dataset(source_path,target_path, max_num_examples=30000):
    ''' This helper method reads from the source and target files to load max_num_examples
    sentences split them into train, development and testing and return relevant data.
    Inputs:
    source_path (string): the full path to the source data, SOURCE_PATH
    target_path (string): the full path to the target data, TARGET_PATH
    Returns:
    train_data (list): a list of 3 elements: source_words, target words, target word labels
    dev_data (list): a list of 2 elements - source words, target word labels
    test_data (list): a list of 2 elements - source words, target word labels
    source_dict (LanguageDict): a LanguageDict object for the source language, Vietnamese.
    target_dict (LanguageDict): a LanguageDict object for the target language, English.
    '''
    # source_lines/target lines are list of strings
    # such that each string is a sentence in the corresponding file
    source_lines = open(source_path).readlines()
    target_lines = open(target_path).readlines()
    assert len(source_lines) == len(target_lines)
    if max_num_examples > 0:
        max_num_examples = min(len(source_lines), max_num_examples)
        source_lines = source_lines[:max_num_examples]
        target_lines = target_lines[:max_num_examples]
    
    # strip trailing/leading whitespaces and tokenize each sentence
    source_sents = [[tok.lower() for tok in sent.strip().split(' ')] for sent in source_lines]

    target_sents = [[tok.lower() for tok in sent.strip().split(' ')] for sent in target_lines]

    # for the target sentences, add <start> and <end> tokens to each sentence
    for sent in target_sents:
        sent.append('<end>')
        sent.insert(0,'<start>')

    
    # create the LanguageDict objects for each file
    source_lang_dict = LanguageDict(source_sents)

    target_lang_dict = LanguageDict(target_sents)

    
    # for the source sentences:
    # we'll use this proportion to split into train/dev/test
    unit = len(source_sents)//10
    # get the sents-as-ids for each sentence
    source_words = [[source_lang_dict.word2ids.get(tok,source_lang_dict.UNK) for tok in sent] for sent in source_sents]

    # 8 parts (80%) of the sentences go to the training data and are padded up to the maximum sentence length
    source_words_train = pad_sequences(source_words[:8*unit])
    
    # 1 part (10%) of the sentences go to the dev data and are padded up to the up to the maximum sentence length
    source_words_dev = pad_sequences(source_words[8*unit:9*unit])
    
    # 1 part (10%) of the sentences go to the test data and are padded up to the up to the maximum sentence length
    source_words_test = pad_sequences(source_words[9*unit:])
    
    
    eos = target_lang_dict.word2ids['<end>']
    # for each sentence, get the word index for the tokens from <start> to up to but not including <end>,
    target_words = [[target_lang_dict.word2ids.get(tok,target_lang_dict.UNK) for tok in sent[:-1]] for sent in target_sents]

    # select the training set and pad the sentences
    target_words_train = pad_sequences(target_words[:8*unit])

    # the label for each target word is the next word, we also add <end> as the last token
    target_words_train_labels = [sent[1:]+[eos] for sent in target_words[:8*unit]]
    
    # pad the labels. Dim = [num_sents, max_sent_length]
    target_words_train_labels = pad_sequences(target_words_train_labels)

    # expand one dimension at the end for the loss computation. Dim = [num_sents, max_sent_length, 1].
    target_words_train_labels = np.expand_dims(target_words_train_labels,axis=2)

    
    
    # get the labels for the dev and test data. No need for inputs here and no need to expand dimensions
    target_words_dev_labels = pad_sequences([sent[1:] + [eos] for sent in target_words[8 * unit:9 * unit]])
    target_words_test_labels = pad_sequences([sent[1:] + [eos] for sent in target_words[9 * unit:]])
    
    # our final data
    train_data = [source_words_train,target_words_train,target_words_train_labels]
    dev_data = [source_words_dev,target_words_dev_labels]
    test_data = [source_words_test,target_words_test_labels]
    
    return train_data,dev_data,test_data,source_lang_dict,target_lang_dict

## The `AttentionLayer` class creates a custom layer for attention

In [ ]:
class AttentionLayer(nn.Module):
    """
    Custom layer implementing rLuong attention.
    """
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def forward(self, encoder_outputs, decoder_outputs, mask_src=None, mask_tgt=None):
        """
        encoder_outputs : [batch_size, max_source_length, hidden_size]
        decoder_outputs : [batch_size, max_target_length, hidden_size]

            1) Tansposing decoder outputs to shape [batch_size, hidden_size, max_target_length]
            2) Doing a batch matrix multiplication with encoder_outputs ->
               shape [batch_size, max_source_length, max_target_length]
            3) Softmax over max_source_length dimension
            4) Weighted sum over encoder_outputs to produce context vectors.
            5) Concatenate them with the original decoder outputs ->
               final shape [batch_size, max_target_length, hidden_size*2]
        """

        # 1) Tansposing decoder outputs
        decoder_outputs_T = torch.permute(decoder_outputs, (0, 2, 1))  # Transpose last two dimensions

        # 2) Batch matrix multiplication with encoder_outputs
        luong_score = torch.matmul(encoder_outputs, decoder_outputs_T)

        if mask_src is not None:
            # Expand mask to match luong_score dimensions: [batch_size, max_source_length, 1] → [batch_size, max_source_length, max_target_length]
            mask_src_expanded = mask_src.unsqueeze(-1).expand_as(luong_score)
            luong_score = luong_score.masked_fill(mask_src_expanded == 0, float('-inf'))
        
        # 3) Softmax 
        luong_score = F.softmax(luong_score, dim=1)

        expanded_luong_score = torch.unsqueeze(luong_score, dim=-1)
        expanded_encoder_outputs = torch.unsqueeze(encoder_outputs, dim=2)

        # 4) Weighted sum 
        weighted_encoder_outputs = expanded_encoder_outputs * expanded_luong_score

        encoder_vector = torch.sum(weighted_encoder_outputs, dim=1)

        # 5) Concatenate 
        new_decoder_outputs = torch.cat([decoder_outputs, encoder_vector], dim=-1)
        return new_decoder_outputs

## NmtModel class `__init__()` method: initialises the network parameters.

This method takes three arguments. The first two are instances of `LanguageDict`, one for the source language (Vietnamese) and one for the target language (English); the third argument is a boolean variable (`use_attention`) that indicates which model (attention/basic) should be used.

In [ ]:
class NmtModel(nn.Module):
    def __init__(self, source_dict, target_dict, use_attention):
        """
        Initializes the NMT Model hyperparameters and layers.
        """
        super().__init__()
        self.source_dict = source_dict
        self.target_dict = target_dict
        self.use_attention = use_attention

        # Hyperparams
        self.hidden_size = 200
        self.embedding_size = 100
        self.hidden_dropout_rate = 0.2
        self.embedding_dropout_rate = 0.2
        self.batch_size = 100
        self.max_target_step = 30

        # Special tokens
        self.SOS = target_dict.word2ids['<start>']
        self.EOS = target_dict.word2ids['<end>']

        # Vocab sizes
        self.vocab_source_size = len(source_dict.vocab)
        self.vocab_target_size = len(target_dict.vocab)

        print(f"number of tokens in source: {self.vocab_source_size}, "
              f"number of tokens in target: {self.vocab_target_size}")

        # Embeddings
        self.embedding_source = nn.Embedding(num_embeddings=self.vocab_source_size, embedding_dim=self.embedding_size, padding_idx=0) 
        self.embedding_target = nn.Embedding(num_embeddings=self.vocab_target_size, embedding_dim=self.embedding_size, padding_idx=0) 

        # Encoder LSTM
        self.encoder_lstm = nn.LSTM(input_size=self.embedding_size, 
                                    hidden_size=self.hidden_size, 
                                    num_layers=1, 
                                    # bias=True, 
                                    batch_first=True, 
                                    dropout=self.hidden_dropout_rate, 
                                    bidirectional=False)  


        # Decoder LSTM
        self.decoder_lstm = nn.LSTM(
            input_size=self.embedding_size,
            hidden_size=self.hidden_size,
            num_layers=1,
            batch_first=True,
            dropout=self.hidden_dropout_rate,
            bidirectional=False
        )

        # Attention (if use_attention)
        if self.use_attention:
            self.decoder_attention = AttentionLayer()

        # Final projection
        # If attention, hidden_size * 2, else hidden_size
        if self.use_attention:
            self.decoder_dense = nn.Linear(self.hidden_size*2, self.vocab_target_size)
        else:
            self.decoder_dense = nn.Linear(self.hidden_size, self.vocab_target_size)

## NmtModel `forward()`, `decode_step()` and `encode()` methods:  builds the PyTorch models for training and inference.

In [ ]:
class NmtModel(NmtModel):
    def forward(self, source_words, target_words):
        """
        Forward pass for training:
          1) Encode the source sentences using the encoder LSTM.
          2) Use the final encoder state to initialize the decoder's hidden state.
          3) Feed all target words into the decoder LSTM in one go (teacher forcing).
          4) (Optional) apply the attention layer between the decoder outputs and the encoder outputs.
          5) Project the decoder outputs to vocabulary logits with self.proj.
        """

        source_mask = (source_words != 0).int()


        # 1) Embedding + dropout
        source_words_embeddings = F.dropout(self.embedding_source(source_words), p=self.embedding_dropout_rate, training=True) 
        target_words_embeddings = F.dropout(self.embedding_target(target_words), p=self.embedding_dropout_rate, training=True) 

        # 2) Encode
        encoder_outputs, (enc_h, enc_c) = self.encoder_lstm(source_words_embeddings) 
        # shape of encoder_outputs = [batch, max_src_len, hidden_size]
        # shape of (enc_h, enc_c) = [1, batch, hidden_size]

        # 3) Decode (teacher forcing)
        # Initialize decoder state with encoder final state
        decoder_outputs, (dec_h, dec_c) = self.decoder_lstm(target_words_embeddings, (enc_h, enc_c)) 
        # shape of decoder_outputs = [batch, max_tgt_len, hidden_size]

        # 4) If use_attention, apply it
        if self.use_attention:
            # decoder_outputs = self.decoder_attention(encoder_outputs, decoder_outputs)
            decoder_outputs = self.decoder_attention(encoder_outputs, decoder_outputs, mask_src=source_mask) 

        # 5) Projection
        decoder_outputs = self.decoder_dense(decoder_outputs)  # [batch, max_tgt_len, vocab_target_size]
        return decoder_outputs

    def decode_step(self, target_words, decoder_states, encoder_outputs):
        """
        A single step of decoder inference:
          - Embedding for the current token
          - One-step LSTM forward
          - (Optional) attention over encoder outputs
          - Project to vocab
        Inputs:
          tgt_input: shape [batch_size, 1]
          decoder_states: (dec_h, dec_c) each is [1, batch_size, hidden_size]
          encoder_outputs: [batch_size, max_src_len, hidden_size]
        Returns:
          logits for the next token, and the new decoder states
        """

        # embedding
        target_words_embeddings = F.dropout(self.embedding_target(target_words), p=self.embedding_dropout_rate, training=False) 
        
        # LSTM step
        decoder_outputs, (dec_h, dec_c) = self.decoder_lstm(target_words_embeddings, decoder_states) 

        if self.use_attention:
            decoder_outputs = self.decoder_attention(encoder_outputs, decoder_outputs) 

        decoder_outputs = self.decoder_dense(decoder_outputs) # [batch, 1, vocab_target_size] 

        
        return decoder_outputs, (dec_h, dec_c)

    def encode(self, source_words):
        """
        Encode the source sequence once for inference.
        """
        source_words_embeddings = F.dropout(self.embedding_source(source_words), p=self.embedding_dropout_rate, training=False)
        encoder_outputs, (enc_h, enc_c) = self.encoder_lstm(source_words_embeddings)
        return encoder_outputs, (enc_h, enc_c)

## NmtModel, `time_used()` method: outputs the time differences between the current time and the input time.

In [12]:
class NmtModel(NmtModel):
  def time_used(self, start_time):
          """
          Outputs the time differences between now and start_time.
          """
          curr_time = time.time()
          used_time = curr_time - start_time
          m = int(used_time // 60)
          s = int(used_time - 60 * m)
          return f"{m} m {s} s"

## The `get_target_sentences()` method takes sentence indices and returns the string tokens.

In [13]:
class NmtModel(NmtModel):
    def get_target_sentences(self, sents, vocab):
        """
        Convert a batch of sequences of token-IDs into strings.
        Stop at <end> or skip <start>.
        """
        str_sents = []
        num_sent, max_len = sents.shape
        for i in range(num_sent):
            str_sent = []
            for j in range(max_len):
                t = int(sents[i, j])
                if t == self.SOS:
                    continue
                if t == self.EOS:
                    break
                str_sent.append(vocab[t])
            str_sents.append(" ".join(str_sent))
        return str_sents

## NmtModel, `eval_process()` method: runs evaluation on the given dataset.

In [14]:
class NmtModel(NmtModel):
    def eval_process(self, dataset):
        """
        Evaluate on a given dataset, returning a BLEU score.
        """
        self.eval()  # set model to eval mode, turning off dropout

        source_words, target_words_labels = dataset
        device = next(self.parameters()).device

        # Convert to torch
        source_words_torch = torch.LongTensor(source_words).to(device)
        target_words_labels_torch = torch.LongTensor(target_words_labels).to(device)

        # 1) Encode
        with torch.no_grad():
            encoder_outputs, (enc_h, enc_c) = self.encode(source_words_torch)

        batch_size = source_words_torch.size(0)
        # Start tokens => shape [batch_size, 1]
        step_tgt = torch.LongTensor([self.SOS]*batch_size).unsqueeze(1).to(device)
        decoder_states = (enc_h, enc_c)

        predictions = []
        # 2) decode up to max_target_step
        for _ in range(self.max_target_step):
            with torch.no_grad():
                logits, decoder_states = self.decode_step(step_tgt, decoder_states, encoder_outputs)
            # argmax over vocab
            step_tgt = torch.argmax(logits, dim=-1)  # [batch_size, 1]
            predictions.append(step_tgt.cpu().numpy())

        # Convert predictions => [batch, max_target_step]
        predictions = np.concatenate(predictions, axis=1)
        # Convert to strings
        candidates = self.get_target_sentences(predictions, self.target_dict.vocab)
        references = self.get_target_sentences(target_words_labels_torch.cpu().numpy(), self.target_dict.vocab)

        # Score with sacrebleu
        score = corpus_bleu(candidates, [references], tokenize='none').score
        print(f"Model BLEU score: {score:.2f}")
        return score

## The `train_main` method starts the training.

In [15]:
class NmtModel(NmtModel):
    def train_model(self, train_data, dev_data, test_data, epochs=10, lr=0.01, clip_norm=5.0, device='cpu'):
        """
        Oversees the training process.
        1) For each epoch, train on the entire training dataset.
        2) Evaluate on dev data after each epoch.
        3) Finally evaluate on test data.
        """
        self.to(device)
        optimizer = optim.Adam(self.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss(ignore_index=self.target_dict.PAD)

        # Unpack data
        source_words_train, target_words_train, target_words_train_labels = train_data
        source_words_dev,   target_words_dev_labels = dev_data
        source_words_test,  target_words_test_labels = test_data

        # For convenience, convert all to torch on CPU first
        source_words_train_torch = torch.LongTensor(source_words_train)
        target_words_train_torch = torch.LongTensor(target_words_train)
        target_words_train_labels_torch = torch.LongTensor(target_words_train_labels.squeeze(-1))  # [batch, max_len]

        # We won't build a fancy DataLoader here; just run with entire batch or smaller mini-batches
        num_samples = source_words_train_torch.size(0)
        idx_list = np.arange(num_samples)

        start_time = time.time()

        for epoch in range(1, epochs+1):
            print(f"Starting training epoch {epoch}/{epochs}")
            epoch_time = time.time()

            # Shuffle data
            np.random.shuffle(idx_list)

            # Mini-batch training
            self.train()  # set model to train mode
            batch_size = self.batch_size
            for start_idx in range(0, num_samples, batch_size):
                end_idx = start_idx + batch_size
                excerpt = idx_list[start_idx:end_idx]

                src_batch = source_words_train_torch[excerpt].to(device)
                tgt_batch = target_words_train_torch[excerpt].to(device)
                tgt_labels_batch = target_words_train_labels_torch[excerpt].to(device)

                optimizer.zero_grad()

                logits = self.forward(src_batch, tgt_batch)  # [batch, tgt_len, vocab_size]

                # Flatten for cross entropy
                # logits: [batch*tgt_len, vocab_size]
                # labels: [batch*tgt_len]
                logits_2d = logits.view(-1, logits.size(-1))
                labels_2d = tgt_labels_batch.view(-1)

                loss = loss_fn(logits_2d, labels_2d)
                loss.backward()

                # Clip gradients
                torch.nn.utils.clip_grad_norm_(self.parameters(), clip_norm)
                optimizer.step()

            print(f"Time used for epoch {epoch}: {self.time_used(epoch_time)}")

            # Evaluate on dev
            print(f"Evaluating on dev set after epoch {epoch}/{epochs}:")
            self.eval_process([source_words_dev, target_words_dev_labels])

        # Training finished
        print("Training finished!")
        print(f"Time used for training: {self.time_used(start_time)}")

        # Evaluate on test set
        print("Evaluating on test set:")
        self.eval_process([source_words_test, target_words_test_labels])

In [16]:
def main(source_path, target_path, use_attention=True):
    max_example = 30000
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print("loading dictionaries...")
    train_data, dev_data, test_data, source_dict, target_dict = load_dataset(
        source_path, target_path, max_num_examples=max_example
    )

    print(f"read {len(train_data[0])}/{len(dev_data[0])}/{len(test_data[0])} train/dev/test batches")

    # Create model
    model = NmtModel(source_dict, target_dict, use_attention=use_attention)
    # print(model)
    # Train
    model.train_model(train_data, dev_data, test_data, epochs=10, lr=0.01, clip_norm=5.0, device=device)

### without attention

In [17]:
main(SOURCE_PATH, TARGET_PATH, use_attention=False)

loading dictionaries...
read 24000/3000/3000 train/dev/test batches
number of tokens in source: 2034, number of tokens in target: 2506


/opt/conda/lib/python3.11/site-packages/torch/nn/modules/rnn.py:88: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


Starting training epoch 1/10


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Time used for epoch 1: 0 m 20 s
Evaluating on dev set after epoch 1/10:
Model BLEU score: 1.42
Starting training epoch 2/10
Time used for epoch 2: 0 m 19 s
Evaluating on dev set after epoch 2/10:


...

Model BLEU score: 3.60
Starting training epoch 10/10


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Time used for epoch 10: 0 m 19 s
Evaluating on dev set after epoch 10/10:
Model BLEU score: 3.62
Training finished!
Time used for training: 3 m 19 s
Evaluating on test set:


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Model BLEU score: 4.12


### with attention

In [18]:
main(SOURCE_PATH, TARGET_PATH, use_attention=True)

loading dictionaries...
read 24000/3000/3000 train/dev/test batches
number of tokens in source: 2034, number of tokens in target: 2506
Starting training epoch 1/10


/opt/conda/lib/python3.11/site-packages/torch/nn/modules/rnn.py:88: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


Time used for epoch 1: 0 m 26 s
Evaluating on dev set after epoch 1/10:


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Model BLEU score: 8.66
Starting training epoch 2/10
Time used for epoch 2: 0 m 20 s
Evaluating on dev set after epoch 2/10:


...

Model BLEU score: 14.32
Starting training epoch 10/10


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Time used for epoch 10: 0 m 25 s
Evaluating on dev set after epoch 10/10:
Model BLEU score: 14.24
Training finished!
Time used for training: 3 m 46 s
Evaluating on test set:


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Model BLEU score: 14.52
